In [ ]:
import pandas as pd
import sqlite3
import os
from pathlib import Path

def sincronizar():
    home = Path.home()
    caminho_excel = home / "Área de trabalho" / "Relatorio_Monitoria.xlsx"
    if not caminho_excel.exists():
        caminho_excel = home / "Área de Trabalho" / "Relatorio_Monitoria.xlsx"

    if not caminho_excel.exists():
        print("❌ Arquivo Excel não encontrado.")
        return

    try:
        df = pd.read_excel(caminho_excel)
        
        # Se o Excel só tem 6 colunas, adicionamos o ID no começo
        if len(df.columns) == 6:
            df.insert(0, 'id', range(1, len(df) + 1))
        
        # FORÇAMOS OS NOMES TÉCNICOS (7 colunas exatas)
        df.columns = ["id", "data", "horario_inicio", "horario_fim", "prof", "min", "desc"]

        # Limpeza de tipos
        df['id'] = pd.to_numeric(df['id'], errors='coerce').fillna(0).astype(int)
        df['min'] = pd.to_numeric(df['min'], errors='coerce').fillna(0).astype(int)
        df['desc'] = df['desc'].fillna("").astype(str)

        conn = sqlite3.connect("meus_registros.db")
        # 'replace' garante que a tabela antiga bugada seja apagada
        df.to_sql('atividades', conn, if_exists='replace', index=False)
        conn.close()
        
        print(f"✅ {len(df)} registros sincronizados com sucesso!")

    except Exception as e:
        print(f"⚠️ Erro: {e}")

if __name__ == "__main__":
    sincronizar()

📊 Colunas detectadas no arquivo: 6
💡 Coluna ID gerada automaticamente.
✅ Sincronia concluída! 11 registros foram salvos no banco.
